# Figures 4, 5 and 6 with BRSET added

Redraws the three figures that need per-image results, now with BRSET (the core 24 retinal models) alongside EyePACS, DDR, HIBA and PAD.
- **Figure 4:** BRSET grade points added; points with fewer than 10 missed cases per model are omitted by one rule applied to every population, and the rank correlation is recomputed.
- **Figure 5:** two rows, agreement on referral decisions (the pre-specified measure) and on score rankings (the recommended measure), for all five populations.
- **Figure 6:** BRSET added.

BRSET is credentialed data: it is read only inside this runtime, and only figures and aggregate numbers are written (`05_figures/manuscript_final/`, `final_figures_data_v2.json`). CPU, High-RAM; about 20 to 30 minutes.

In [1]:
# Cell 1: load both domains
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
OUT = f'{BASE}/05_figures/manuscript_final'; os.makedirs(OUT, exist_ok=True)
def pick(df, c): return next(x for x in c if x in df.columns)
def tag(k): return '' if k == 0 else f'_seed{k}'
PAPER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
         'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']
CK = {'densenet121': 'densenet', 'efficientnet_b3': 'effnet', 'resnet50': 'resnet', 'vit_base': 'vit', 'retfound': 'retfound'}
ck_of = lambda v: next(c for p, c in CK.items() if v.startswith(p))
def load(path, ids, col=None):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = col or pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
DOM = {}
fr = pd.read_csv(f'{BASE}/01_splits/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
dd = pd.read_csv(f'{BASE}/01_splits/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
for site, df, tpl, f in (('EyePACS', fr, 'eyepacs_predictions_fresh_{m}_v2.csv', 0.25), ('DDR', dd, 'ddr_predictions_{m}_v2.csv', None)):
    ids = df.image_id.to_numpy(); g = df[pick(df, ['level','grade'])].to_numpy().astype(int)
    M = {f'{v}{tag(k)}': {'s': load(f'{BASE}/03_predictions/' + tpl.format(m=f'{v}{tag(k)}'), ids), 'ckpt': ck_of(v), 'variant': v} for v in PAPER for k in (0, 1, 2)}
    DOM[site] = {'domain': 'retina', 'models': M, 'pos': g >= 2, 'classes': {'moderate': g == 2, 'severe NPDR': g == 3, 'PDR': g == 4},
                 'f': f if f else 1.31 * float((dd.grade[dd.half == 'A'] >= 2).mean())}
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True); ids = D.image_id.to_numpy()
    M = {}
    for s in ('HAM10000', 'BCN20000'):
        for a in ('resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2'):
            for k in (0, 1): M[f'{s}_{a}_seed{k}'] = {'s': load(f'{DERM}/predictions/{es}__{s}_{a}_seed{k}.csv', ids, 'p_malignant'), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy()
    DOM[es] = {'domain': 'dermatology', 'models': M, 'pos': mal, 'classes': {'BCC': (D.dx == 'BCC').to_numpy(), 'SCC': (D.dx == 'SCC').to_numpy(), 'melanoma': (D.dx == 'MEL').to_numpy()},
               'f': 1.31 * mal.mean()}
# BRSET (credentialed data: stays in this runtime; only figures and aggregate numbers are written)
import hashlib
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
assert sha(f'{BR}/label_brset.csv') == json.load(open(f'{BR}/brset_access_record.json'))['label_file_sha256']
L = pd.read_csv(f'{BR}/label_brset.csv', dtype=str); gB = pd.to_numeric(L['DR_ICDR'], errors='coerce')
L = L[gB.isin([0, 1, 2, 3, 4])].copy(); L['grade'] = gB[gB.isin([0, 1, 2, 3, 4])].astype(int)
L['image_id'] = L['image_id'].astype(str).str.replace(r'\.(jpg|jpeg|png)$', '', regex=True); L = L.sort_values('image_id').reset_index(drop=True)
M = {}
for s in ('eyepacs', 'ddr'):
    for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class'):
        for k in (0, 1, 2):
            M[f'{s}_{a}{tag(k)}'] = {'s': load(f'{BASE}/03_predictions/brset_predictions_{s}_{a}{tag(k)}_v2.csv', L.image_id.to_numpy()), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
gb = L.grade.to_numpy()
DOM['BRSET'] = {'domain': 'retina', 'models': M, 'pos': gb >= 2, 'classes': {'moderate': gb == 2, 'severe NPDR': gb == 3, 'PDR': gb == 4}, 'f': 1.31 * float((gb >= 2).mean())}
for k, v in DOM.items(): print(k, len(v['models']), 'models, f =', round(v['f'], 3), {c: int(m.sum()) for c, m in v['classes'].items()})

Mounted at /content/drive
EyePACS 33 models, f = 0.25 {'moderate': 2373, 'severe NPDR': 384, 'PDR': 328}
DDR 33 models, f = 0.555 {'moderate': 4477, 'severe NPDR': 236, 'PDR': 913}
HIBA 24 models, f = 0.609 {'BCC': 340, 'SCC': 158, 'melanoma': 253}
PAD 24 models, f = 0.62 {'BCC': 839, 'SCC': 190, 'melanoma': 52}
BRSET 24 models, f = 0.074 {'moderate': 451, 'severe NPDR': 78, 'PDR': 396}


In [2]:
# Cell 2: per ordered pair and class: partner's own miss rate, union rescue share, matched-volume rescue share; pair relationship and referral agreement
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
def relation(site, A, B):
    if 'source' not in A:
        if A['ckpt'] != B['ckpt']: return 'different design'
        return 'seed sibling' if A['variant'] == B['variant'] else 'variant sibling'
    if A['variant'] == B['variant']: return 'seed sibling'
    if A['ckpt'] == B['ckpt']: return 'same checkpoint, different data'
    return 'different design, same data' if A['source'] == B['source'] else 'different design, different data'
ROWS = []
for site, S in DOM.items():
    M = S['models']; names = list(M); n = len(S['pos'])
    R = {m: refer(M[m]['s'], S['f']) for m in names}; RK = {m: rankdata(M[m]['s']) / n for m in names}
    for a, b in itertools.permutations(names, 2):
        e = refer(np.maximum(RK[a], RK[b]), S['f']); rel = relation(site, M[a], M[b]); kap = kappa(R[a], R[b])
        for c, y in S['classes'].items():
            ma, mb = y & ~R[a], y & ~R[b]; na = ma.sum()
            if na == 0: continue
            ROWS.append({'site': site, 'domain': S['domain'], 'a': a, 'b': b, 'relationship': rel, 'kappa': kap, 'class': c,
                         'partner_miss_rate': mb.sum() / y.sum(), 'anchor_miss_rate': na / y.sum(),
                         'union_rescue_share': (ma & ~mb).sum() / na, 'mv_rescue_share': (na - (y & ~e).sum()) / na,
                         'lift': ((ma & mb).sum() / y.sum()) / ((na / y.sum()) * (mb.sum() / y.sum())) if mb.sum() else np.nan})
PR = pd.DataFrame(ROWS); print(PR.groupby(['site', 'class']).size().to_string())

site     class      
BRSET    PDR             552
         moderate        552
         severe NPDR     529
DDR      PDR            1056
         moderate       1056
         severe NPDR     960
EyePACS  PDR            1056
         moderate       1056
         severe NPDR    1056
HIBA     BCC             552
         SCC             552
         melanoma        552
PAD      BCC             552
         SCC             552
         melanoma        552


In [3]:
# Cell 4: shared style
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, SKY, ORANGE, VERM, PINK, GREEN, GREY = '#0072B2', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7', '#009E73', '#666666'
def save(fig, name): fig.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)
def title(fig, t): fig.suptitle(t, x=0.01, ha='left', fontsize=11, weight='bold')


In [4]:
# Cell 4: Figure 4, the miss-rate law (condition-population points with fewer than 10 misses per model omitted) (rescue share vs partner's own miss rate for that class; both domains)
CCOL = {'moderate': '#56B4E9', 'severe NPDR': '#0072B2', 'PDR': '#003A70', 'BCC': '#E69F00', 'SCC': '#CC79A7', 'melanoma': '#D55E00'}
MARK = {'EyePACS': 'o', 'DDR': 's', 'BRSET': 'P', 'HIBA': '^', 'PAD': 'D'}
PR['anchor_misses'] = PR.anchor_miss_rate * PR.apply(lambda r: int(DOM[r.site]['classes'][r['class']].sum()), axis=1)
ND0 = PR[PR.relationship.str.startswith('different design')]
MED = ND0.groupby(['site', 'class']).anchor_misses.median()
KEEP = MED[MED >= 10].index; DROPPED = [f'{s} {c} ({m:.1f})' for (s, c), m in MED[MED < 10].items()]
ND = ND0[ND0.set_index(['site', 'class']).index.isin(KEEP)]
print('omitted (median misses per model < 10):', DROPPED)
SUMM = ND.groupby(['site', 'domain', 'class']).agg(partner_miss_rate=('partner_miss_rate', 'median'), union_rescue=('union_rescue_share', 'median'),
                                                  mv_rescue=('mv_rescue_share', 'median'), lift=('lift', 'median'), n_pairs=('a', 'size')).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ax = axes[0]; smp = ND.sample(min(len(ND), 6000), random_state=1)
for c, col in CCOL.items():
    s = smp[smp['class'] == c]; ax.scatter(s.partner_miss_rate, s.union_rescue_share, s=4, alpha=0.18, color=col)
p = np.linspace(0.005, 0.6, 200)
for L, ls in ((2, ':'), (5, '--'), (10, '-.')): ax.plot(p, np.clip(1 - L * p, 0, 1), ls, color='#555', lw=1, label=f'1 - {L} x miss rate')
ax.set_xlabel("Partner's own miss rate for that class"); ax.set_ylabel("Share of deployed model's misses the partner catches")
ax.set_title('a  Every pair and class (different designs); curves: theory for lift 2, 5, 10', loc='left', fontsize=9.5, weight='bold'); ax.legend(fontsize=7.5, frameon=False)
ax = axes[1]
for _, r in SUMM.iterrows():
    ax.scatter(r.partner_miss_rate, 100 * r.mv_rescue, s=110, marker=MARK[r.site], color=CCOL[r['class']], edgecolor='#222', lw=0.6, zorder=3)
    ax.annotate(f"{r['class']} ({r.site})", (r.partner_miss_rate, 100 * r.mv_rescue), textcoords='offset points', xytext=(6, 3), fontsize=7)
rho = spearmanr(SUMM.partner_miss_rate, SUMM.mv_rescue)[0]
ax.set_xlabel("Median per-model miss rate for that class"); ax.set_ylabel('Median % of misses rescued at matched volume')
ax.set_title(f'b  Rescue follows how rarely each reader misses (rho {rho:+.2f}), not severity', loc='left', fontsize=9.5, weight='bold')
title(fig, f'Figure 4. A second reader rescues most where each reader alone rarely misses (retina and skin; {len(SUMM)} condition-population points)')
fig.tight_layout(); save(fig, 'Fig4_miss_rate_law')
print(SUMM.round(3).to_string())

omitted (median misses per model < 10): ['BRSET severe NPDR (5.0)', 'DDR severe NPDR (2.0)']
saved Fig4_miss_rate_law
       site       domain        class  partner_miss_rate  union_rescue  mv_rescue   lift  n_pairs
0     BRSET       retina          PDR              0.266         0.451      0.097  1.788      432
1     BRSET       retina     moderate              0.427         0.286      0.057  1.398      432
2       DDR       retina          PDR              0.053         0.560      0.250  8.669      864
3       DDR       retina     moderate              0.173         0.435      0.118  3.324      864
4   EyePACS       retina          PDR              0.055         0.650      0.400  7.130      864
5   EyePACS       retina     moderate              0.351         0.316      0.076  1.963      864
6   EyePACS       retina  severe NPDR              0.068         0.456      0.167  8.234      864
7      HIBA  dermatology          BCC              0.062         0.810      0.308  3.333      480


In [5]:
# Cell 5: Figure 5, siblings: agreement on referral decisions (top) and on score rankings (bottom), five populations
from sklearn.metrics import roc_auc_score
PK = PR.drop_duplicates(['site', 'a', 'b'])[['site', 'a', 'b', 'relationship', 'kappa']]; PK = PK[PK.a < PK.b].copy()
RANKS = {}
for site, S in DOM.items():
    n = len(S['pos']); RK = {m: rankdata(S['models'][m]['s']) / n for m in S['models']}
    RANKS[site] = {(a, b): float(np.corrcoef(RK[a], RK[b])[0, 1]) for a, b in itertools.combinations(sorted(S['models']), 2)}
PK['rank_corr'] = [RANKS[r.site][(r.a, r.b)] for r in PK.itertuples()]
SIBREL = lambda r: r.str.contains('sibling')
ORDER = ['seed sibling', 'variant sibling', 'same checkpoint, different data', 'different design, same data', 'different design, different data', 'different design']
RCOL = {'seed sibling': '#D55E00', 'variant sibling': '#E69F00', 'same checkpoint, different data': '#CC79A7', 'different design, same data': '#56B4E9', 'different design, different data': '#0072B2', 'different design': '#0072B2'}
DET = {}; sites = list(DOM)
fig, axes = plt.subplots(2, len(sites), figsize=(4.6 * len(sites), 8.8))
for row, (metric, lab) in enumerate((('kappa', 'Agreement on referral decisions (kappa)'), ('rank_corr', 'Agreement on score rankings (rank correlation)'))):
    for ax, site in zip(axes[row], sites):
        s = PK[PK.site == site]; sib = SIBREL(s.relationship); dif = s.relationship.str.startswith('different design')
        thr = float(np.percentile(s[metric][dif], 95)); flag = float((s[metric][sib] > thr).mean())
        auc = float(roc_auc_score(np.r_[np.ones(sib.sum()), np.zeros(dif.sum())], np.r_[s[metric][sib], s[metric][dif]]))
        DET.setdefault(site, {})[metric] = {'auc_siblings_vs_different_design': round(auc, 3), 'relative_threshold': round(thr, 4), 'siblings_flagged': round(flag, 3)}
        rels = [r for r in ORDER if r in set(s.relationship)]
        bp = ax.boxplot([s[s.relationship == r][metric] for r in rels], vert=False, patch_artist=True, widths=0.6, labels=[r.replace(', ', ',\n') for r in rels])
        for patch, r in zip(bp['boxes'], rels): patch.set_facecolor(RCOL[r]); patch.set_alpha(0.55)
        ax.axvline(thr, color=GREEN, lw=1.2)
        ax.set_title(f"{site}: AUC {auc:.2f}; flagged {100 * flag:.0f}%", loc='left', fontsize=9, weight='bold'); ax.set_xlabel(lab, fontsize=8); ax.tick_params(axis='y', labelsize=7)
title(fig, 'Figure 5. Related systems give themselves away. Green line: relative rule (95th percentile among systems known to differ); top, referral decisions; bottom, score rankings')
fig.tight_layout(); save(fig, 'Fig5_siblings'); print(json.dumps(DET, indent=1))

/tmp/ipykernel_7207/111809606.py:21: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([s[s.relationship == r][metric] for r in rels], vert=False, patch_artist=True, widths=0.6, labels=[r.replace(', ', ',\n') for r in rels])
/tmp/ipykernel_7207/111809606.py:21: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([s[s.relationship == r][metric] for r in rels], vert=False, patch_artist=True, widths=0.6, labels=[r.replace(', ', ',\n') for r in rels])
/tmp/ipykernel_7207/111809606.py:21: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([s[s.relationship == r][metric] for r in rels], vert

saved Fig5_siblings
{
 "EyePACS": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.5979,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6886,
   "siblings_flagged": 1.0
  }
 },
 "DDR": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6478,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.8423,
   "siblings_flagged": 1.0
  }
 },
 "HIBA": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.947,
   "relative_threshold": 0.6856,
   "siblings_flagged": 0.833
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 0.985,
   "relative_threshold": 0.8244,
   "siblings_flagged": 0.917
  }
 },
 "PAD": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.981,
   "relative_threshold": 0.4646,
   "siblings_flagged": 0.75
  },
  "rank_corr": {
   "auc_siblings_vs_different_

In [6]:
# Cell 6: Figure 6, agreement predicts benefit (matched-volume rescue of the most important class, by kappa band, all four sets)
KEYC = {'EyePACS': 'PDR', 'DDR': 'PDR', 'HIBA': 'melanoma', 'PAD': 'BCC'}
BANDS = [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6)); BEN = {}
for site, S in DOM.items():
    s = PR[(PR.site == site) & (PR['class'].isin(list(S['classes'])))]
    allc = s.groupby(pd.cut(s.kappa, BANDS)).mv_rescue_share.agg(['median', 'count']).reset_index()
    allc = allc[allc['count'] >= 10]; mid = [iv.mid for iv in allc.kappa]
    axes[0].plot(mid, 100 * allc['median'], '-', marker=MARK[site], lw=2, label=f'{site} (all target classes)')
    BEN[site] = {str(k): {'median_rescue': round(float(m), 3), 'pairs': int(c)} for k, m, c in zip(allc.kappa, allc['median'], allc['count'])}
axes[0].set_xlabel('Referral agreement with the deployed model (kappa)'); axes[0].set_ylabel('Median % of misses rescued at matched volume')
axes[0].set_title('a  The more a candidate agrees, the less it adds', loc='left', fontsize=9.5, weight='bold'); axes[0].legend(fontsize=7.5, frameon=False)
rels = PR.groupby(['site', 'relationship']).mv_rescue_share.median().unstack(0)
rels = rels.reindex([r for r in ORDER if r in rels.index])
rels.mul(100).plot.barh(ax=axes[1], color=['#0072B2', '#56B4E9', '#009E73', '#E69F00', '#D55E00'][:rels.shape[1]], width=0.75)
axes[1].set_xlabel('Median % of misses rescued at matched volume'); axes[1].set_ylabel('')
axes[1].set_title('b  Siblings add least; different data and design add most', loc='left', fontsize=9.5, weight='bold'); axes[1].legend(fontsize=7.5, frameon=False)
title(fig, 'Figure 6. Agreement on local unlabelled images predicts how much a candidate can add')
fig.tight_layout(); save(fig, 'Fig6_agreement_benefit')
json.dump({'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'status': 'exploratory synthesis of saved predictions',
           'miss_rate_law_summary': SUMM.round(4).to_dict(orient='records'), 'miss_rate_law_spearman': round(float(rho), 3),
           'sibling_detector': DET, 'omitted_points': DROPPED, 'benefit_by_kappa_band': BEN,
           'rescue_by_relationship': rels.round(4).to_dict()}, open(f'{OUT}/final_figures_data_v2.json', 'w'), indent=2, default=str)
drive.flush_and_unmount(); drive.mount('/content/drive')
print('Wrote 05_figures/manuscript_final: Fig4, Fig5, Fig6 and final_figures_data_v2.json')

/tmp/ipykernel_7207/2399224734.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  allc = s.groupby(pd.cut(s.kappa, BANDS)).mv_rescue_share.agg(['median', 'count']).reset_index()
/tmp/ipykernel_7207/2399224734.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  allc = s.groupby(pd.cut(s.kappa, BANDS)).mv_rescue_share.agg(['median', 'count']).reset_index()
/tmp/ipykernel_7207/2399224734.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence

saved Fig6_agreement_benefit
Mounted at /content/drive
Wrote 05_figures/manuscript_final: Fig4, Fig5, Fig6 and final_figures_data_v2.json
